Tentative Plan:


1.   Collect 1–3 public Gen Z/slang datasets.
2.   Normalize them into one input-output format.
3.   Fine-tune BART.
4.   Build 2 baselines.
5.   Create your own 100–200 sentence test set.
6.   Evaluate with human judgments on aspects like: meaning preserved, sounds Gen Z, sounds natural



## STEP 1

In [1]:
!pip -q install datasets pandas pyarrow

In [2]:
import pandas as pd
from datasets import load_dataset, Dataset

pd.set_option("display.max_colwidth", 200)

In [3]:
# Load the three Hugging Face datasets

# 1) Main slang lexicon / glossary dataset
ds_mlb = load_dataset("MLBtrio/genz-slang-dataset")

# 2) Paired sentence rewrite dataset
ds_pairs_1k = load_dataset("Programmer-RD-AI/genz-slang-pairs-1k")

# 3) Small paired translation dataset
ds_sherry = load_dataset("thesherrycode/gen-z-slangs-translation")

print(ds_mlb)
print(ds_pairs_1k)
print(ds_sherry)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

all_slangs.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1779 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

genz_dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1005 [00:00<?, ? examples/s]

gen_z_slangs_translation.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/140 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Slang', 'Description', 'Example', 'Context'],
        num_rows: 1779
    })
})
DatasetDict({
    train: Dataset({
        features: ['normal', 'gen_z'],
        num_rows: 1005
    })
})
DatasetDict({
    train: Dataset({
        features: ['Plain English', 'Gen-Z Slang'],
        num_rows: 140
    })
})


In [4]:
# Inspect schemas
print("MLBtrio columns:", ds_mlb["train"].column_names)
print("Pairs-1k columns:", ds_pairs_1k["train"].column_names)
print("Sherry columns:", ds_sherry["train"].column_names)

print("\nSample from MLBtrio:")
print(ds_mlb["train"][0])

print("\nSample from Pairs-1k:")
print(ds_pairs_1k["train"][0])

print("\nSample from Sherry:")
print(ds_sherry["train"][0])

MLBtrio columns: ['Slang', 'Description', 'Example', 'Context']
Pairs-1k columns: ['normal', 'gen_z']
Sherry columns: ['Plain English', 'Gen-Z Slang']

Sample from MLBtrio:
{'Slang': 'W', 'Description': 'Shorthand for win', 'Example': 'Got the job today, big W!', 'Context': 'Typically used in conversations to celebrate success, achievements, or positive outcomes'}

Sample from Pairs-1k:
{'normal': "I'm really tired today, I think I need some rest.", 'gen_z': "I'm totally drained today, need to catch some Z's."}

Sample from Sherry:
{'Plain English': 'That’s amazing!', 'Gen-Z Slang': 'That’s lit!'}


In [5]:
# Normalize the two parallel datasets into one format, one schema
#   source | plain | gen_z

pairs_1k_df = pd.DataFrame(ds_pairs_1k["train"])
pairs_1k_df = pairs_1k_df.rename(columns={
    "normal": "plain",
    "gen_z": "gen_z"
})
pairs_1k_df["source"] = "Programmer-RD-AI/genz-slang-pairs-1k"

sherry_df = pd.DataFrame(ds_sherry["train"])
sherry_df = sherry_df.rename(columns={
    "Plain English": "plain",
    "Gen-Z Slang": "gen_z"
})
sherry_df["source"] = "thesherrycode/gen-z-slangs-translation"

parallel_df = pd.concat(
    [
        pairs_1k_df[["source", "plain", "gen_z"]],
        sherry_df[["source", "plain", "gen_z"]],
    ],
    ignore_index=True
)

# basic cleanup
parallel_df = parallel_df.dropna()
parallel_df["plain"] = parallel_df["plain"].astype(str).str.strip()
parallel_df["gen_z"] = parallel_df["gen_z"].astype(str).str.strip()

# remove exact duplicates
parallel_df = parallel_df.drop_duplicates(subset=["plain", "gen_z"]).reset_index(drop=True)

print("Combined parallel dataset shape:", parallel_df.shape)
parallel_df.head(20)

Combined parallel dataset shape: (1144, 3)


,source,plain,gen_z
0,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some Z's."
1,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I just want to relax at home.","I'm hella drained today, just wanna chill at home."
2,Programmer-RD-AI/genz-slang-pairs-1k,I'm really excited for the concert tonight.,I'm so hype for the concert tonight.
3,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some coffee.","I'm so drained today, I gotta get me some caffeine."
4,Programmer-RD-AI/genz-slang-pairs-1k,I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to turn up!"
5,Programmer-RD-AI/genz-slang-pairs-1k,I'm really tired today and just want to relax at home.,I'm mad tired today and just wanna chill at home.
6,Programmer-RD-AI/genz-slang-pairs-1k,I'm just hanging out with my friends tonight.,I'm just vibing with the squad tonight.
7,Programmer-RD-AI/genz-slang-pairs-1k,I'm going to grab some coffee before work.,"I'm tryna hit up some coffee before I clock in, fr."
8,Programmer-RD-AI/genz-slang-pairs-1k,"Hey, did you see the new movie last night? It was really good.","Yo, peep the new flick last night? It was lit."
9,Programmer-RD-AI/genz-slang-pairs-1k,I'm just chilling at home and watching some TV.,"Just vibing at home, Netflix and chillin'."


In [6]:
# Normalize the MLBtrio slang lexicon separately
mlb_df = pd.DataFrame(ds_mlb["train"])

# Normalize names if they exist exactly as expected
# Expected columns are: Slang, Description, Example, Context

expected_cols = ["Slang", "Description", "Example", "Context"]
missing = [c for c in expected_cols if c not in mlb_df.columns]
if missing:
    print("Warning: missing expected MLBtrio columns:", missing)

mlb_lexicon_df = mlb_df.copy()
mlb_lexicon_df["source"] = "MLBtrio/genz-slang-dataset"

# keep the most useful columns if present
keep_cols = [c for c in ["source", "Slang", "Description", "Example", "Context"] if c in mlb_lexicon_df.columns]
mlb_lexicon_df = mlb_lexicon_df[keep_cols].dropna(subset=["Slang"]).reset_index(drop=True)

print("MLB lexicon shape:", mlb_lexicon_df.shape)
mlb_lexicon_df.head(20)

MLB lexicon shape: (1779, 5)


,source,Slang,Description,Example,Context
0,MLBtrio/genz-slang-dataset,W,Shorthand for win,"Got the job today, big W!","Typically used in conversations to celebrate success, achievements, or positive outcomes"
1,MLBtrio/genz-slang-dataset,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.","Often used when referring to a failure or mishap, either personally or generally"
2,MLBtrio/genz-slang-dataset,L+ratio,Response to a comment or action on the internet that is particularly bad.,Your tweet got 5 likes and 100 replies calling you out. L + ratio.,Popularized on social media platforms to signify that someone not only failed but also had their failure amplified through backlash.
3,MLBtrio/genz-slang-dataset,Dank,excellent or of very high quality,That meme is so dank!,Commonly used in internet slang to refer to memes or humor that are edgy or particularly funny.
4,MLBtrio/genz-slang-dataset,Cheugy,Derogatory term for Millennials. Used when millennials are perceived to be excessively attempting to be trendy or stylish.,"That phrase is so cheugy, no one says that anymore.","Used to refer to things that were once popular but are now considered passé, usually in a millennial vs. Gen Z context."
5,MLBtrio/genz-slang-dataset,TFW,That feeling when,TFW you finish a big project and can finally relax,"Often paired with an image or caption to convey an emotion or experience, mostly used in memes."
6,MLBtrio/genz-slang-dataset,Woke,being politically aware,He’s really woke about climate change.,"Originally used in activist circles, it has since been co-opted and sometimes used sarcastically."
7,MLBtrio/genz-slang-dataset,Bop,An excellent song or album.,This new track is a total bop!,"Commonly used in reference to music, especially upbeat songs that are hard to resist dancing to."
8,MLBtrio/genz-slang-dataset,G.O.A.T,The greatest of all time,Michael Jordan is the GOAT of basketball.,Frequently used in sports but has expanded to any field where someone is considered the best.
9,MLBtrio/genz-slang-dataset,Smol,"Something that is small, and in most cases exceptionally adorable.","Look at that smol puppy, it’s adorable!","Typically used when describing pets, babies, or anything tiny and endearing."


In [7]:
# Create a model-ready Hugging Face Dataset object
hf_parallel = Dataset.from_pandas(parallel_df)

print(hf_parallel)
print(hf_parallel[0])

Dataset({
    features: ['source', 'plain', 'gen_z'],
    num_rows: 1144
})
{'source': 'Programmer-RD-AI/genz-slang-pairs-1k', 'plain': "I'm really tired today, I think I need some rest.", 'gen_z': "I'm totally drained today, need to catch some Z's."}


In [8]:
print("Total paired examples:", len(parallel_df))
print("Examples by source:")
print(parallel_df["source"].value_counts())

print("\nRandom paired samples:")
display(parallel_df.sample(min(10, len(parallel_df)), random_state=42))

print("\nRandom slang lexicon samples:")
display(mlb_lexicon_df.sample(min(10, len(mlb_lexicon_df)), random_state=42))

Total paired examples: 1144
Examples by source:
source
Programmer-RD-AI/genz-slang-pairs-1k      1005
thesherrycode/gen-z-slangs-translation     139
Name: count, dtype: int64

Random paired samples:


,source,plain,gen_z
218,Programmer-RD-AI/genz-slang-pairs-1k,I'm so tired after studying all night.,I'm hella drained after pulling an all-nighter.
809,Programmer-RD-AI/genz-slang-pairs-1k,"Hey, I just got back from the store and grabbed some snacks.","Yo, just hit the store and stacked up on snacks, fr fr!"
501,Programmer-RD-AI/genz-slang-pairs-1k,"Hey, I just finished my homework and I’m feeling pretty good.","Yo, just peeped my homework done and I’m vibing hard."
649,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to chill at home all day today.,"I'm just gonna veg out at home all day today, no cap."
323,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to relax and watch some videos this afternoon.,I'm just gonna chill and scroll through vids today.
506,Programmer-RD-AI/genz-slang-pairs-1k,I'm feeling really tired today after staying up late yesterday.,I'm mad drained today 'cause I stayed up all night yesterday.
1005,thesherrycode/gen-z-slangs-translation,That’s amazing!,That’s lit!
107,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to relax at home and watch some TV.,I'm just chillin' at home and vibing with some Netflix.
731,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to relax and watch some videos later.,"I'm just chillin' and vibin' with some vids later, fr."
170,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to grab some coffee before heading to work.,I'm finna sip some java before I dip to work.



Random slang lexicon samples:


,source,Slang,Description,Example,Context
1192,MLBtrio/genz-slang-dataset,NR,Nice roll,"You got double sixes, NR!",A compliment used in dice games when someone gets a good result.
426,MLBtrio/genz-slang-dataset,BAK,Back at keyboard,"BRB, BAK in 5.",Used to inform someone that you are back at your computer after being away.
1474,MLBtrio/genz-slang-dataset,SWALK,Sealed with a loving kiss,The letter had SWALK written on the envelope.,"A more affectionate version of SWAK, often used in love letters."
765,MLBtrio/genz-slang-dataset,FOMO,Fear of missing out,"I have such FOMO, everyone’s going to that concert except me.",Refers to the feeling of anxiety or regret over missing an event or experience.
485,MLBtrio/genz-slang-dataset,BIOYN,Blow it out your nose,"You’re being ridiculous, BIOYN.",A less harsh but still dismissive insult.
479,MLBtrio/genz-slang-dataset,BIF,Before I forget,"BIF, don’t forget the meeting at 3!",Used to quickly mention something before it slips the person’s mind.
109,MLBtrio/genz-slang-dataset,ick,used to express disgust at something unpleasant or offensive.,"He talks with his mouth full, that gave me the ick.",Frequently used in dating scenarios to describe a moment that turns someone off.
774,MLBtrio/genz-slang-dataset,FU,Freak you,"You messed up my plans, FU!",A harsh or angry way to express frustration or anger towards someone.
1431,MLBtrio/genz-slang-dataset,SMAZED,Smoky haze,The sky was SMAZED after the fireworks.,"Describes a smoky or hazy atmosphere, often caused by smoke."
1188,MLBtrio/genz-slang-dataset,NOYB,None of your business,What’s going on with you? NOYB!,A blunt way to tell someone to stay out of your personal matters.


In [9]:
# Save clean CSVs so GitHub/deployment is easier later
parallel_df.to_csv("genz_parallel_combined.csv", index=False)
mlb_lexicon_df.to_csv("genz_slang_lexicon.csv", index=False)

print("Saved:")
print("- genz_parallel_combined.csv")
print("- genz_slang_lexicon.csv")

Saved:
- genz_parallel_combined.csv
- genz_slang_lexicon.csv


In [10]:
# Build a lightweight lexical substitution map
slang_terms = mlb_lexicon_df["Slang"].dropna().astype(str).str.strip().unique().tolist()

print("Number of unique slang terms:", len(slang_terms))
print("Sample slang terms:", slang_terms[:50])

Number of unique slang terms: 1588
Sample slang terms: ['W', 'L', 'L+ratio', 'Dank', 'Cheugy', 'TFW', 'Woke', 'Bop', 'G.O.A.T', 'Smol', 'Fam', 'Glow up', 'Stan', 'Ghosting', 'Salty', 'Sip tea', 'Drip', 'Iykyk', 'Rent free', 'Catch these hands', 'Drag', 'Bussin’', 'Snatched', 'Cancel culture', 'Ffs', 'e-boy', 'e-girl', 'Big yikes', 'Finna', 'cap', 'High-key', 'Simp', 'Camp', 'Snack', 'Take several seats', 'Sheesh', 'Hits different', 'Bet', 'Periodt', 'Finesse', 'I’m weak', 'Main character', 'Sis', 'Sending me', 'Gaup', 'This ain’t it, chief', 'Extra', 'Clapback', '@ me', 'Asl']


## Step 2

In [11]:
!pip -q install transformers accelerate sentencepiece scikit-learn

import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [12]:
# Reload the cleaned paired dataset
# Otherwise load from CSV
try:
    parallel_df
    print("Using parallel_df already in memory.")
except NameError:
    parallel_df = pd.read_csv("genz_parallel_combined.csv")
    print("Loaded parallel_df from CSV.")

print(parallel_df.shape)
parallel_df.head()

Using parallel_df already in memory.
(1144, 3)


,source,plain,gen_z
0,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some Z's."
1,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I just want to relax at home.","I'm hella drained today, just wanna chill at home."
2,Programmer-RD-AI/genz-slang-pairs-1k,I'm really excited for the concert tonight.,I'm so hype for the concert tonight.
3,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some coffee.","I'm so drained today, I gotta get me some caffeine."
4,Programmer-RD-AI/genz-slang-pairs-1k,I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to turn up!"


In [13]:
# Build bidirectional dataset
forward_df = parallel_df.copy()
forward_df["direction"] = "standard_to_genz"
forward_df["input_text"] = "translate to Gen Z: " + forward_df["plain"]
forward_df["target_text"] = forward_df["gen_z"]

reverse_df = parallel_df.copy()
reverse_df["direction"] = "genz_to_standard"
reverse_df["input_text"] = "translate to standard English: " + reverse_df["gen_z"]
reverse_df["target_text"] = reverse_df["plain"]

bidirectional_df = pd.concat([forward_df, reverse_df], ignore_index=True)

bidirectional_df = bidirectional_df[
    ["source", "direction", "plain", "gen_z", "input_text", "target_text"]
].reset_index(drop=True)

print("Original parallel_df shape:", parallel_df.shape)
print("Bidirectional dataset shape:", bidirectional_df.shape)

Original parallel_df shape: (1144, 3)
Bidirectional dataset shape: (2288, 6)


In [14]:
# Add model input prompt, frame task clearly for BART
parallel_df["input_text"] = "translate to Gen Z: " + parallel_df["plain"]
parallel_df["target_text"] = parallel_df["gen_z"]

parallel_df[["input_text", "target_text"]].head(10)

,input_text,target_text
0,"translate to Gen Z: I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some Z's."
1,"translate to Gen Z: I'm really tired today, I just want to relax at home.","I'm hella drained today, just wanna chill at home."
2,translate to Gen Z: I'm really excited for the concert tonight.,I'm so hype for the concert tonight.
3,"translate to Gen Z: I'm really tired today, I think I need some coffee.","I'm so drained today, I gotta get me some caffeine."
4,translate to Gen Z: I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to turn up!"
5,translate to Gen Z: I'm really tired today and just want to relax at home.,I'm mad tired today and just wanna chill at home.
6,translate to Gen Z: I'm just hanging out with my friends tonight.,I'm just vibing with the squad tonight.
7,translate to Gen Z: I'm going to grab some coffee before work.,"I'm tryna hit up some coffee before I clock in, fr."
8,"translate to Gen Z: Hey, did you see the new movie last night? It was really good.","Yo, peep the new flick last night? It was lit."
9,translate to Gen Z: I'm just chilling at home and watching some TV.,"Just vibing at home, Netflix and chillin'."


In [15]:
# Train/Validation split, 90/10 for now
train_df, val_df = train_test_split(
    bidirectional_df,
    test_size=0.10,
    random_state=42,
    shuffle=True,
    stratify=bidirectional_df["direction"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("\nTrain direction counts:")
print(train_df["direction"].value_counts())
print("\nValidation direction counts:")
print(val_df["direction"].value_counts())


Train direction counts:
direction
standard_to_genz    1030
genz_to_standard    1029
Name: count, dtype: int64

Validation direction counts:
direction
genz_to_standard    115
standard_to_genz    114
Name: count, dtype: int64


In [16]:
# Convert to Hugging Face DatasetDict
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds
})

print(dataset_dict)
print(dataset_dict["train"][0])

DatasetDict({
    train: Dataset({
        features: ['source', 'direction', 'plain', 'gen_z', 'input_text', 'target_text'],
        num_rows: 2059
    })
    validation: Dataset({
        features: ['source', 'direction', 'plain', 'gen_z', 'input_text', 'target_text'],
        num_rows: 229
    })
})
{'source': 'Programmer-RD-AI/genz-slang-pairs-1k', 'direction': 'genz_to_standard', 'plain': "I'm planning to just chill at home this weekend.", 'gen_z': "I'm just vibing at home all weekend, no cap.", 'input_text': "translate to standard English: I'm just vibing at home all weekend, no cap.", 'target_text': "I'm planning to just chill at home this weekend."}


In [17]:
# Load BART tokenizer, start from facebook/bart-base, adjust in later steps
checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

print("Tokenizer loaded:", checkpoint)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: facebook/bart-base


In [18]:
# tokenization settings
max_input_length = 64
max_target_length = 64

In [19]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names
)

print(tokenized_datasets)
print(tokenized_datasets["train"][0].keys())

Map:   0%|          | 0/2059 [00:00<?, ? examples/s]

Map:   0%|          | 0/229 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2059
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 229
    })
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [20]:
# inspect a decoded example
sample_idx = 0

raw_example = dataset_dict["train"][sample_idx]
tok_example = tokenized_datasets["train"][sample_idx]

print("RAW INPUT:")
print(raw_example["input_text"])
print("\nRAW TARGET:")
print(raw_example["target_text"])

print("\nTOKENIZED INPUT DECODED:")
print(tokenizer.decode(tok_example["input_ids"], skip_special_tokens=True))

label_ids = [x for x in tok_example["labels"] if x != tokenizer.pad_token_id]
print("\nTOKENIZED TARGET DECODED:")
print(tokenizer.decode(label_ids, skip_special_tokens=True))

RAW INPUT:
translate to standard English: I'm just vibing at home all weekend, no cap.

RAW TARGET:
I'm planning to just chill at home this weekend.

TOKENIZED INPUT DECODED:
translate to standard English: I'm just vibing at home all weekend, no cap.

TOKENIZED TARGET DECODED:
I'm planning to just chill at home this weekend.


In [21]:
# Save processed artifacts for later
bidirectional_df.to_csv("genz_bidirectional_combined.csv", index=False)
train_df.to_csv("genz_train_split.csv", index=False)
val_df.to_csv("genz_val_split.csv", index=False)

tokenized_datasets.save_to_disk("genz_tokenized_bart_dataset")

print("Saved:")
print("- genz_bidrectional_combined.csv")
print("- genz_train_split.csv")
print("- genz_val_split.csv")
print("- genz_tokenized_bart_dataset/")

Saving the dataset (0/1 shards):   0%|          | 0/2059 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/229 [00:00<?, ? examples/s]

Saved:
- genz_bidrectional_combined.csv
- genz_train_split.csv
- genz_val_split.csv
- genz_tokenized_bart_dataset/


## Step 3

In [22]:
!pip -q install transformers accelerate evaluate rouge_score
import re
import random
import numpy as np
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [23]:
import torch
print(torch.cuda.get_device_name(0))

Tesla T4


In [24]:
# Setting up rule-based slang baseline, very simple
# A lightweight starter dictionary.
# Can expand this later using the MLB lexicon or custom mappings.
slang_map = {

    # Greetings
    r"\bhello\b": "yo",
    r"\bhi\b": "yo",
    r"\bhey\b": "yo",
    r"\bhello there\b": "yo fr",

    # People
    r"\bfriend\b": "bestie",
    r"\bfriends\b": "besties",
    r"\bgroup\b": "crew",
    r"\bpeople\b": "peeps",
    r"\beveryone\b": "y'all",

    # Agreement / affirmation
    r"\bI agree\b": "facts",
    r"\bthat is true\b": "no cap",
    r"\btrue\b": "fr",
    r"\byes\b": "bet",
    r"\bof course\b": "bet",
    r"\bokay\b": "bet",
    r"\bok\b": "bet",

    # Negation
    r"\bno\b": "nah",
    r"\bnot really\b": "lowkey nah",
    r"\bI do not know\b": "idk",
    r"\bI don't know\b": "idk",

    # Intensifiers
    r"\bvery\b": "lowkey",
    r"\breally\b": "actually",
    r"\bextremely\b": "mad",
    r"\bso much\b": "hella",
    r"\ba lot\b": "hella",

    # Positive sentiment
    r"\bgreat\b": "fire",
    r"\bgood\b": "solid",
    r"\bamazing\b": "bussin",
    r"\bawesome\b": "fire",
    r"\bexcellent\b": "elite",
    r"\bnice\b": "clean",
    r"\bperfect\b": "flawless",
    r"\bfun\b": "a vibe",
    r"\benjoyable\b": "a vibe",

    # Negative sentiment
    r"\bbad\b": "mid",
    r"\bterrible\b": "trash",
    r"\bawful\b": "trash",
    r"\bboring\b": "mid",
    r"\bannoying\b": "mad annoying",
    r"\bweird\b": "sus",

    # Emotions
    r"\bexcited\b": "hyped",
    r"\btired\b": "drained",
    r"\bexhausted\b": "wiped",
    r"\bembarrassing\b": "cringe",
    r"\bembarrassed\b": "lowkey embarrassed",
    r"\bhappy\b": "vibing",
    r"\bsad\b": "down bad",
    r"\bangry\b": "pressed",

    # Actions / verbs
    r"\bgoing\b": "finna",
    r"\bwant to\b": "wanna",
    r"\bgoing to\b": "gonna",
    r"\btrying to\b": "tryna",
    r"\blet me\b": "lemme",

    # Social actions
    r"\bhang out\b": "chill",
    r"\brelax\b": "chill",
    r"\bspend time\b": "kick it",
    r"\bmeet\b": "link",

    # Communication
    r"\btell me\b": "lmk",
    r"\blet me know\b": "lmk",
    r"\bmessage me\b": "hit me up",
    r"\bcall me\b": "hit me up",

    # Time / context
    r"\bright now\b": "rn",
    r"\btoday\b": "today fr",
    r"\btonight\b": "tonight fr",
    r"\blater\b": "later fr",

    # Expressions
    r"\bI am not lying\b": "no cap",
    r"\bno lie\b": "no cap",
    r"\bfor real\b": "fr",
    r"\bseriously\b": "deadass",

    # Food
    r"\bdelicious\b": "bussin",
    r"\btasty\b": "bussin",

    # Situations
    r"\bthat is a problem\b": "that's tough",
    r"\bthat is unfortunate\b": "that's tough",

    # Win/Loss
    r"\bwin\b": "W",
    r"\bloss\b": "L",
    r"\bthat is a big win\b": "that's a W",
    r"\bthat is a loss\b": "that's an L",

    # Judgment
    r"\bimpressive\b": "valid",
    r"\brespectable\b": "valid",
    r"\bdisappointing\b": "mid",

    # Slang nouns
    r"\bmoney\b": "bag",
    r"\bcar\b": "whip",
    r"\bphone\b": "phone fr",

    # Style / appearance
    r"\blooks good\b": "clean",
    r"\blooks great\b": "fire",
    r"\bdressed well\b": "drippy",

    # Internet slang
    r"\bI am weak\b": "I'm dead",
    r"\bthat is funny\b": "I'm dead",
    r"\bso funny\b": "I'm crying",

    # Agreement phrases
    r"\bI understand\b": "I get it fr",
    r"\bI see\b": "I see fr",

    # Apologies
    r"\bI am sorry\b": "my bad",
    r"\bI’m sorry\b": "my bad",
    r"\bsorry\b": "my bad",

    # Requests
    r"\bcan you\b": "can you lowkey",
    r"\bcould you\b": "could you lowkey",

    # Misc
    r"\bthink about\b": "vibe on",
    r"\bconsider\b": "vibe on",
    r"\bunderstand\b": "get",

    # Extra Gen Z phrases
    r"\bthis is different\b": "this hits different",
    r"\bthis is unique\b": "this hits different",

    # Social media
    r"\bwatch videos\b": "scroll vids",
    r"\bwatch youtube\b": "binge YouTube",

    # Energy expressions
    r"\bfull of energy\b": "amped",
    r"\bno energy\b": "low energy fr",

    # Mild exaggeration
    r"\bvery good\b": "so fire",
    r"\bvery bad\b": "so mid",

    # Confidence
    r"\bI am confident\b": "I'm locked in",
    r"\bconfident\b": "locked in",

    # Uncertainty
    r"\bmaybe\b": "lowkey maybe",
    r"\bperhaps\b": "lowkey",

    # Extra phrases
    r"\bwhat are you doing\b": "wyd",
    r"\bhow are you\b": "how you doing fr",
    r"\bsee you later\b": "catch you later",
}

def rule_based_genz(text, max_replacements=3, apply_prob=1, add_suffix_prob=0):
    out = text.strip()
    replacements_made = 0

    # Apply longer / more specific patterns first
    sorted_rules = sorted(slang_map.items(), key=lambda x: -len(x[0]))

    for pattern, replacement in sorted_rules:
        if replacements_made >= max_replacements:
            break

        if re.search(pattern, out, flags=re.IGNORECASE):
            if random.random() <= apply_prob:
                new_out, num_subs = re.subn(
                    pattern,
                    replacement,
                    out,
                    count=1,  # only replace first match for stability
                    flags=re.IGNORECASE
                )
                if num_subs > 0 and new_out != out:
                    out = new_out
                    replacements_made += 1

    # Light cleanup / contraction pass
    cleanup_rules = [
        (r"\bI am\b", "I'm"),
        (r"\bI will\b", "I'll"),
        (r"\bdo not\b", "don't"),
        (r"\bcannot\b", "can't"),
        (r"\bcan not\b", "can't"),
        (r"\bgoing to\b", "gonna"),
        (r"\bwant to\b", "wanna"),
        (r"\blet me\b", "lemme"),
    ]

    for pattern, replacement in cleanup_rules:
        out = re.sub(pattern, replacement, out, flags=re.IGNORECASE)

    # Normalize extra spaces
    out = re.sub(r"\s+", " ", out).strip()

    # Make sure punctuation exists
    if not re.search(r"[.!?]$", out):
        out += "."

    # Optional suffix injection, but do it gently
    if add_suffix_prob > 0 and random.random() <= add_suffix_prob:
        if re.search(r"[.!?]$", out):
            out = re.sub(r"[.!?]$", " fr.", out)

    return out

In [25]:
# Quick test of the rule-based baseline
test_sentences = [
    "I am very excited for the concert tonight.",
    "My friend is really tired today.",
    "That was very embarrassing.",
    "I do not agree with that.",
    "This food is amazing."
]

for s in test_sentences:
    print("PLAIN: ", s)
    print("RULE:  ", rule_based_genz(s))
    print()

PLAIN:  I am very excited for the concert tonight.
RULE:   I'm lowkey hyped for the concert tonight fr.

PLAIN:  My friend is really tired today.
RULE:   My bestie is actually drained today.

PLAIN:  That was very embarrassing.
RULE:   That was lowkey cringe.

PLAIN:  I do not agree with that.
RULE:   I don't agree with that.

PLAIN:  This food is amazing.
RULE:   This food is bussin.



In [26]:
# Reload tokenizer and model
checkpoint = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

print("Loaded model and tokenizer:", checkpoint)

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loaded model and tokenizer: facebook/bart-base


In [27]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [28]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_genz_model",
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=4,
    predict_with_generate=False,
    bf16=True,
    fp16=False,
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to="none"
)

In [29]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
50,4.894460
100,0.576891
150,0.253086
200,0.217634
250,0.198466
300,0.151951
350,0.162278
400,0.152250
450,0.145555
500,0.142332


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1032, training_loss=0.38939664229866144, metrics={'train_runtime': 344.5762, 'train_samples_per_second': 23.902, 'train_steps_per_second': 2.995, 'total_flos': 313861841879040.0, 'train_loss': 0.38939664229866144, 'epoch': 4.0})

In [30]:
# Save final model
save_dir = "./final_bart_genz_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Saved model and tokenizer to {save_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to ./final_bart_genz_model


In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def translate_text(text, direction="standard_to_genz", max_new_tokens=50, num_beams=4):
    if direction == "standard_to_genz":
        prompt = f"translate to Gen Z: {text}"
    elif direction == "genz_to_standard":
        prompt = f"translate to standard English: {text}"
    else:
        raise ValueError("direction must be 'standard_to_genz' or 'genz_to_standard'")

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Backward-compatible wrapper so old code still works
def genz_translate(text, max_new_tokens=50):
    return translate_text(
        text,
        direction="standard_to_genz",
        max_new_tokens=max_new_tokens
    )

In [32]:
sample_df = val_df.sample(min(15, len(val_df)), random_state=42)

for _, row in sample_df.iterrows():
    source_text = row["plain"]
    ref_text = row["gen_z"]
    pred_text = genz_translate(source_text)

    print("PLAIN: ", source_text)
    print("GEN Z: ", ref_text)
    print("MODEL: ", pred_text)
    print("-" * 70)

PLAIN:  I'm really tired today, I just want to relax and do nothing.
GEN Z:  I'm so drained today, just wanna chill and do nada.
MODEL:  I'm hella drained today, just tryna chill and do nothing.
----------------------------------------------------------------------
PLAIN:  I'm just really tired today and all I want to do is relax at home.
GEN Z:  I'm totally drained today and just wanna chill at home, fr.
MODEL:  I'm hella drained today and all I wanna do is chill at home.
----------------------------------------------------------------------
PLAIN:  Hey, want to grab some coffee later?
GEN Z:  Yo, down to get some coffee later?
MODEL:  Yo, wanna snag some coffee later?
----------------------------------------------------------------------
PLAIN:  I'm going to grab some coffee. Do you want to come?
GEN Z:  I'm finna get some coffee, yolo? Do you wanna roll?
MODEL:  I'm finna snag some coffee. You tryna slide?
----------------------------------------------------------------------
PLAIN:

In [33]:
bidirectional_test_examples = [
    ("I am really tired after work.", "standard_to_genz"),
    ("That movie was amazing.", "standard_to_genz"),
    ("Do you want to hang out later?", "standard_to_genz"),
    ("I'm hella drained after work, no cap.", "genz_to_standard"),
    ("That flick was straight fire.", "genz_to_standard"),
    ("Yo, wanna link later?", "genz_to_standard"),
]

for text, direction in bidirectional_test_examples:
    print("DIRECTION:", direction)
    print("INPUT:    ", text)
    print("OUTPUT:   ", translate_text(text, direction=direction))
    print("-" * 70)

DIRECTION: standard_to_genz
INPUT:     I am really tired after work.
OUTPUT:    I'm totally drained after grindin' all day.
----------------------------------------------------------------------
DIRECTION: standard_to_genz
INPUT:     That movie was amazing.
OUTPUT:    That flick was straight fire.
----------------------------------------------------------------------
DIRECTION: standard_to_genz
INPUT:     Do you want to hang out later?
OUTPUT:    Yo, wanna chill later?
----------------------------------------------------------------------
DIRECTION: genz_to_standard
INPUT:     I'm hella drained after work, no cap.
OUTPUT:    I'm really tired after work.
----------------------------------------------------------------------
DIRECTION: genz_to_standard
INPUT:     That flick was straight fire.
OUTPUT:    That movie was really good.
----------------------------------------------------------------------
DIRECTION: genz_to_standard
INPUT:     Yo, wanna link later?
OUTPUT:    Hey, do you want

## Step 4 (WIP)

In [34]:
# Compare rule-based baseline vs BART

# Make sure model is in eval mode
model.eval()

comparison_sample_df = val_df.sample(min(25, len(val_df)), random_state=42).reset_index(drop=True)

comparison_rows = []

for _, row in comparison_sample_df.iterrows():
    input_text = row["plain"]
    reference_text = row["gen_z"]

    rule_output = rule_based_genz(input_text)
    bart_output = genz_translate(input_text)

    comparison_rows.append({
        "input_text": input_text,
        "reference_text": reference_text,
        "rule_output": rule_output,
        "bart_output": bart_output
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head(10)

,input_text,reference_text,rule_output,bart_output
0,"I'm really tired today, I just want to relax and do nothing.","I'm so drained today, just wanna chill and do nada.","I'm actually drained today, I just wanna relax and do nothing.","I'm hella drained today, just tryna chill and do nothing."
1,I'm just really tired today and all I want to do is relax at home.,"I'm totally drained today and just wanna chill at home, fr.",I'm just actually drained today and all I wanna do is relax at home.,I'm hella drained today and all I wanna do is chill at home.
2,"Hey, want to grab some coffee later?","Yo, down to get some coffee later?","yo, wanna grab some coffee later fr?","Yo, wanna snag some coffee later?"
3,I'm going to grab some coffee. Do you want to come?,"I'm finna get some coffee, yolo? Do you wanna roll?",I'm gonna grab some coffee. Do you wanna come?,I'm finna snag some coffee. You tryna slide?
4,I'm so tired after a long day at work.,"I'm so drained after a mega long shift, fr.",I'm so drained after a long day at work.,I'm totally drained after a long day at the grind.
5,I'm just trying to get through the day and stay positive.,I'm just vibing through the day and keeping it chill.,I'm just tryna get through the day and stay positive.,I'm just vibing through the day and keeping it chill.
6,I'm really excited about the concert tomorrow.,I'm super hyped for the concert tomorrow!,I'm actually hyped about the concert tomorrow.,I'm super hyped for the concert tomorrow!
7,"I'm really tired today, so I think I'm just going to chill at home.","I'm so drained today, might just vibe solo and stay in.","I'm actually drained today, so I think I'm just gonna chill at home.","I'm hella drained today, so I'm just gonna chill at home."
8,It’s very loud.,It’s deafening.,It’s lowkey loud.,It’s deafening.
9,I'm going to hang out with my friends at the park later.,"I'm vibing with my crew at the park later, fr.!",I'm gonna chill with my besties at the park later.,"I'm finna chill with my squad at the park later, no cap."


In [35]:
for _, row in comparison_df.head(10).iterrows():
    print("INPUT:      ", row["input_text"])
    print("REFERENCE:  ", row["reference_text"])
    print("RULE:       ", row["rule_output"])
    print("BART:       ", row["bart_output"])
    print("-" * 80)

INPUT:       I'm really tired today, I just want to relax and do nothing.
REFERENCE:   I'm so drained today, just wanna chill and do nada.
RULE:        I'm actually drained today, I just wanna relax and do nothing.
BART:        I'm hella drained today, just tryna chill and do nothing.
--------------------------------------------------------------------------------
INPUT:       I'm just really tired today and all I want to do is relax at home.
REFERENCE:   I'm totally drained today and just wanna chill at home, fr.
RULE:        I'm just actually drained today and all I wanna do is relax at home.
BART:        I'm hella drained today and all I wanna do is chill at home.
--------------------------------------------------------------------------------
INPUT:       Hey, want to grab some coffee later?
REFERENCE:   Yo, down to get some coffee later?
RULE:        yo, wanna grab some coffee later fr?
BART:        Yo, wanna snag some coffee later?
------------------------------------------------

In [36]:
# Manual evaluation template
manual_eval_df = comparison_df.copy()

manual_eval_df["rule_meaning"] = None
manual_eval_df["rule_style"] = None
manual_eval_df["rule_fluency"] = None

manual_eval_df["bart_meaning"] = None
manual_eval_df["bart_style"] = None
manual_eval_df["bart_fluency"] = None

manual_eval_df["overall_preference"] = None
manual_eval_df["notes"] = None

#manual_eval_df.head(10)

In [37]:
#manual_eval_df.to_csv("manual_eval_template.csv", index=False)
#print("Saved manual_eval_template.csv")

In [38]:
# Manual eval sample
eval_sample_df = comparison_df.head(10).copy().reset_index(drop=True)

for i, row in eval_sample_df.iterrows():
    print(f"\nExample {i+1}")
    print("INPUT:      ", row["input_text"])
    print("REFERENCE:  ", row["reference_text"])
    print("RULE:       ", row["rule_output"])
    print("BART:       ", row["bart_output"])
    print("-" * 80)


Example 1
INPUT:       I'm really tired today, I just want to relax and do nothing.
REFERENCE:   I'm so drained today, just wanna chill and do nada.
RULE:        I'm actually drained today, I just wanna relax and do nothing.
BART:        I'm hella drained today, just tryna chill and do nothing.
--------------------------------------------------------------------------------

Example 2
INPUT:       I'm just really tired today and all I want to do is relax at home.
REFERENCE:   I'm totally drained today and just wanna chill at home, fr.
RULE:        I'm just actually drained today and all I wanna do is relax at home.
BART:        I'm hella drained today and all I wanna do is chill at home.
--------------------------------------------------------------------------------

Example 3
INPUT:       Hey, want to grab some coffee later?
REFERENCE:   Yo, down to get some coffee later?
RULE:        yo, wanna grab some coffee later fr?
BART:        Yo, wanna snag some coffee later?
---------------

## Step 5 Multi-candidate generation for reranking


In [61]:
# Step 5A: Multi-candidate generation for reranking

def generate_candidates(
    text,
    direction="standard_to_genz",
    num_return_sequences=5,
    num_beams=8,
    max_new_tokens=50,
    diversity_penalty=0.8,
    no_repeat_ngram_size=2
):
    """
    Generate multiple candidate translations from BART for later reranking.
    Uses beam search with diverse outputs.
    """

    if direction == "standard_to_genz":
        prompt = f"translate to Gen Z: {text}"
    elif direction == "genz_to_standard":
        prompt = f"translate to standard English: {text}"
    else:
        raise ValueError("direction must be 'standard_to_genz' or 'genz_to_standard'")

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )

    decoded = [
        tokenizer.decode(output, skip_special_tokens=True).strip()
        for output in outputs
    ]

    # remove exact duplicates while preserving order
    unique_candidates = list(dict.fromkeys(decoded))

    return unique_candidates

In [62]:
# Step 5A-2: Sampling-based candidate generation for more diverse reranking options

def generate_candidates_sampling(
    text,
    direction="standard_to_genz",
    num_return_sequences=5,
    max_new_tokens=50,
    temperature=0.9,
    top_k=50,
    top_p=0.92,
    no_repeat_ngram_size=2
):
    """
    Generate multiple diverse candidate translations using sampling.
    This should give the reranker more meaningful options than plain beam search.
    """

    if direction == "standard_to_genz":
        prompt = f"translate to Gen Z: {text}"
    elif direction == "genz_to_standard":
        prompt = f"translate to standard English: {text}"
    else:
        raise ValueError("direction must be 'standard_to_genz' or 'genz_to_standard'")

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            num_beams=1,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=no_repeat_ngram_size
        )

    decoded = [
        tokenizer.decode(output, skip_special_tokens=True).strip()
        for output in outputs
    ]

    unique_candidates = list(dict.fromkeys(decoded))
    return unique_candidates

In [63]:
def compare_candidate_generators(text, direction="standard_to_genz", num_return_sequences=5):
    print("=" * 100)
    print("DIRECTION:", direction)
    print("INPUT:", text)

    print("\nBEAM CANDIDATES:")
    beam_cands = generate_candidates(
        text,
        direction=direction,
        num_return_sequences=num_return_sequences
    )
    for i, cand in enumerate(beam_cands, start=1):
        print(f"{i}. {cand}")

    print("\nSAMPLING CANDIDATES:")
    sample_cands = generate_candidates_sampling(
        text,
        direction=direction,
        num_return_sequences=num_return_sequences
    )
    for i, cand in enumerate(sample_cands, start=1):
        print(f"{i}. {cand}")

    return beam_cands, sample_cands

In [64]:
test_examples = [
    ("I am really tired after work.", "standard_to_genz"),
    ("That movie was amazing.", "standard_to_genz"),
    ("Do you want to hang out later?", "standard_to_genz"),
    ("I'm hella drained after work, no cap.", "genz_to_standard")
]

for text, direction in test_examples:
    compare_candidate_generators(
        text,
        direction=direction,
        num_return_sequences=5
    )
    print("\n")

DIRECTION: standard_to_genz
INPUT: I am really tired after work.

BEAM CANDIDATES:
1. I'm totally drained after grindin' all day.
2. I'm totally drained after work, fr fr.
3. I'm totally wiped after grindin' all day.
4. I'm hella drained after work, fr fr.
5. I'm totally drained after work, fr.

SAMPLING CANDIDATES:
1. I'm totally drained after grinding all day.
2. I'm deadass drained after grindin' all day.
3. I'm hella drained after grinding all day.
4. I'm totally drained after grindin' all day.
5. I'm hella drained after grindin' all day, no cap.


DIRECTION: standard_to_genz
INPUT: That movie was amazing.

BEAM CANDIDATES:
1. That flick was straight fire.
2. That flick was fire.
3. That flick was fire, no cap.
4. That flick was mad sick.
5. That flick was lit.

SAMPLING CANDIDATES:
1. That flick was fire, no cap.
2. That flick was fire.
3. That flick was straight fire.
4. That flick was lit.


DIRECTION: standard_to_genz
INPUT: Do you want to hang out later?

BEAM CANDIDATES:
1. Y

In [65]:
# Step 5A-3: Hybrid candidate generation (beam + sampling)

def generate_candidates_hybrid(
    text,
    direction="standard_to_genz",
    beam_return_sequences=5,
    sample_return_sequences=5,
    num_beams=8,
    max_new_tokens=50,
    temperature=0.9,
    top_k=50,
    top_p=0.92,
    no_repeat_ngram_size=2
):
    """
    Combine beam-search and sampling candidates into one pool.
    This gives the reranker both safe and diverse options.
    """

    beam_candidates = generate_candidates(
        text=text,
        direction=direction,
        num_return_sequences=beam_return_sequences,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens,
        no_repeat_ngram_size=no_repeat_ngram_size
    )

    sample_candidates = generate_candidates_sampling(
        text=text,
        direction=direction,
        num_return_sequences=sample_return_sequences,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        no_repeat_ngram_size=no_repeat_ngram_size
    )

    combined = beam_candidates + sample_candidates

    # remove duplicates while preserving order
    unique_candidates = list(dict.fromkeys(combined))

    return unique_candidates

In [66]:
# Quick test of candidate generation
test_examples = [
    ("I am really tired after work.", "standard_to_genz"),
    ("That movie was amazing.", "standard_to_genz"),
    ("Do you want to hang out later?", "standard_to_genz"),
    ("I'm hella drained after work, no cap.", "genz_to_standard")
]

for text, direction in test_examples:
    print("=" * 80)
    print("DIRECTION:", direction)
    print("INPUT:", text)
    print()

    candidates = generate_candidates(
        text,
        direction=direction,
        num_return_sequences=5
    )

    for i, cand in enumerate(candidates, start=1):
        print(f"{i}. {cand}")
    print()

DIRECTION: standard_to_genz
INPUT: I am really tired after work.

1. I'm totally drained after grindin' all day.
2. I'm totally drained after work, fr fr.
3. I'm totally wiped after grindin' all day.
4. I'm hella drained after work, fr fr.
5. I'm totally drained after work, fr.

DIRECTION: standard_to_genz
INPUT: That movie was amazing.

1. That flick was straight fire.
2. That flick was fire.
3. That flick was fire, no cap.
4. That flick was mad sick.
5. That flick was lit.

DIRECTION: standard_to_genz
INPUT: Do you want to hang out later?

1. Yo, wanna chill later?
2. Yo, you tryna chill later?
3. Yo, wanna vibe later?
4. Yo, you tryna vibe later?
5. Yo, you down to chill later?

DIRECTION: genz_to_standard
INPUT: I'm hella drained after work, no cap.

1. I'm really tired after work.
2. I'm just really tired after work.
3. I'm really tired after work today.
4. I'm feeling really tired after work.
5. I'm really tired after work, no cap.



In [ ]:
import re
from collections import Counter

In [70]:
# Step 5B-2: Semantic similarity helper for reranking
from sentence_transformers import SentenceTransformer, util

# load once
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_similarity(source, candidate):
    embeddings = semantic_model.encode([source, candidate], convert_to_tensor=True)
    sim = util.cos_sim(embeddings[0], embeddings[1]).item()
    return float(sim)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [71]:
# Simple heuristic reranker
GENZ_MARKERS = {
    "fr", "frfr", "fr fr", "no cap", "cap", "hella",
    "lowkey", "highkey", "valid", "mid", "fire", "lit",
    "ate", "slaps", "bet", "tryna", "wanna", "yo",
    "wild", "crazy", "deadass", "sus", "rizz", "ick",
    "drip", "bussin", "straight", "vibe", "vibes",
    "type shit", "it s giving", "it giving", "hits different",
    "down bad", "locked in", "cooked", "goated", "tea",
    "shook", "pressed", "extra", "outta pocket", "for real"
}

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def token_set(text):
    return set(normalize_text(text).split())

def slang_score(text):
    t = normalize_text(text)
    score = 0
    for marker in GENZ_MARKERS:
        if marker in t:
            score += 1
    return score

def jaccard_similarity(a, b):
    a_set = token_set(a)
    b_set = token_set(b)
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)

def repetition_penalty(text):
    tokens = normalize_text(text).split()
    if not tokens:
        return 0.0
    counts = Counter(tokens)
    repeated = sum(c - 1 for c in counts.values() if c > 1)
    return repeated / len(tokens)

def length_ratio_score(source, candidate):
    src_len = max(1, len(normalize_text(source).split()))
    cand_len = max(1, len(normalize_text(candidate).split()))
    ratio = cand_len / src_len

    # best if somewhat close in length
    return max(0.0, 1.0 - abs(1.0 - ratio))

def score_candidate(source, candidate, direction="standard_to_genz"):
    """
    Returns a score and component breakdown for one candidate.
    Improved version with semantic similarity:
    - rewards meaning preservation
    - rewards style, but not too aggressively
    - penalizes near-copies
    - rewards moderate rewriting
    - penalizes repetition
    - keeps length reasonable
    """

    copy_sim = jaccard_similarity(source, candidate)
    rep_pen = repetition_penalty(candidate)
    len_score = length_ratio_score(source, candidate)
    slang = slang_score(candidate)
    sem_sim = semantic_similarity(source, candidate)

    slang_norm = min(slang / 2.0, 1.0)

    if 0.35 <= copy_sim <= 0.75:
        rewrite_reward = 1.0
    elif 0.20 <= copy_sim < 0.35:
        rewrite_reward = 0.7
    elif 0.75 < copy_sim <= 0.90:
        rewrite_reward = 0.4
    else:
        rewrite_reward = 0.1

    if copy_sim >= 0.85:
        copy_penalty = 1.0
    elif copy_sim >= 0.70:
        copy_penalty = 0.7
    elif copy_sim >= 0.50:
        copy_penalty = 0.4
    else:
        copy_penalty = 0.0

    if direction == "standard_to_genz":
        score = (
            0.30 * sem_sim +
            0.22 * slang_norm +
            0.18 * rewrite_reward +
            0.15 * len_score -
            0.10 * copy_penalty -
            0.05 * rep_pen
        )

    elif direction == "genz_to_standard":
        standardness = 1.0 - slang_norm

        score = (
            0.35 * sem_sim +
            0.22 * standardness +
            0.18 * rewrite_reward +
            0.15 * len_score -
            0.07 * copy_penalty -
            0.03 * rep_pen
        )

    else:
        raise ValueError("direction must be 'standard_to_genz' or 'genz_to_standard'")

    return {
        "candidate": candidate,
        "score": score,
        "semantic_similarity": sem_sim,
        "slang_score": slang,
        "slang_norm": slang_norm,
        "copy_similarity": copy_sim,
        "copy_penalty": copy_penalty,
        "rewrite_reward": rewrite_reward,
        "repetition_penalty": rep_pen,
        "length_score": len_score
    }

In [72]:
# Rank candidates and select the best one
def rerank_candidates(source, candidates, direction="standard_to_genz"):
    scored = [
        score_candidate(source, cand, direction=direction)
        for cand in candidates
    ]
    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored

def translate_with_reranking(
    text,
    direction="standard_to_genz",
    candidate_mode="hybrid",   # "beam", "sampling", or "hybrid"
    num_return_sequences=5,
    num_beams=8,
    max_new_tokens=50,
    temperature=0.9,
    top_k=50,
    top_p=0.92
):
    if candidate_mode == "beam":
        candidates = generate_candidates(
            text=text,
            direction=direction,
            num_return_sequences=num_return_sequences,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens
        )

    elif candidate_mode == "sampling":
        candidates = generate_candidates_sampling(
            text=text,
            direction=direction,
            num_return_sequences=num_return_sequences,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )

    elif candidate_mode == "hybrid":
        candidates = generate_candidates_hybrid(
            text=text,
            direction=direction,
            beam_return_sequences=num_return_sequences,
            sample_return_sequences=num_return_sequences,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )

    else:
        raise ValueError("candidate_mode must be 'beam', 'sampling', or 'hybrid'")

    ranked = rerank_candidates(text, candidates, direction=direction)
    best = ranked[0]["candidate"]

    return {
        "input": text,
        "direction": direction,
        "candidate_mode": candidate_mode,
        "best_candidate": best,
        "ranked_candidates": ranked
    }

In [73]:
# Compare beam vs sampling vs hybrid reranking

test_examples = [
    ("I am really tired after work.", "standard_to_genz"),
    ("That movie was amazing.", "standard_to_genz"),
    ("Do you want to hang out later?", "standard_to_genz"),
    ("I'm hella drained after work, no cap.", "genz_to_standard")
]

for text, direction in test_examples:
    print("=" * 100)
    print("INPUT:", text)
    print("DIRECTION:", direction)

    for mode in ["beam", "sampling", "hybrid"]:
        result = translate_with_reranking(
            text,
            direction=direction,
            candidate_mode=mode,
            num_return_sequences=5
        )
        print(f"\nMODE: {mode}")
        print("BEST:", result["best_candidate"])

    print()

INPUT: I am really tired after work.
DIRECTION: standard_to_genz

MODE: beam
BEST: I'm totally drained after work, fr fr.

MODE: sampling
BEST: I'm dead tired after work, fr fr.

MODE: hybrid
BEST: I'm totally drained after work, fr fr.

INPUT: That movie was amazing.
DIRECTION: standard_to_genz

MODE: beam
BEST: That flick was fire.

MODE: sampling
BEST: That flick was fire.

MODE: hybrid
BEST: That flick was fire.

INPUT: Do you want to hang out later?
DIRECTION: standard_to_genz

MODE: beam
BEST: Yo, you down to chill later?

MODE: sampling
BEST: Yo, you tryna chill later?

MODE: hybrid
BEST: Yo, you down to chill later?

INPUT: I'm hella drained after work, no cap.
DIRECTION: genz_to_standard

MODE: beam
BEST: I'm really tired after work today.

MODE: sampling
BEST: I'm really tired after work.

MODE: hybrid
BEST: I'm really tired after work today.



In [74]:
# Reranker test
test_examples = [
    ("I am really tired after work.", "standard_to_genz"),
    ("That movie was amazing.", "standard_to_genz"),
    ("Do you want to hang out later?", "standard_to_genz"),
    ("I'm hella drained after work, no cap.", "genz_to_standard")
]

for text, direction in test_examples:
    print("=" * 90)
    print("DIRECTION:", direction)
    print("INPUT:", text)

    result = translate_with_reranking(
        text,
        direction=direction,
        num_return_sequences=5
    )

    print("\nBEST CANDIDATE:")
    print(result["best_candidate"])

    print("\nRANKED CANDIDATES:")
    for i, item in enumerate(result["ranked_candidates"], start=1):
        print(
            f"{i}. {item['candidate']} | "
            f"score={item['score']:.3f}, "
            f"sem={item['semantic_similarity']:.3f}, "
            f"slang={item['slang_score']}, "
            f"slang_norm={item['slang_norm']:.3f}, "
            f"copy={item['copy_similarity']:.3f}, "
            f"copy_pen={item['copy_penalty']:.3f}, "
            f"rewrite={item['rewrite_reward']:.3f}, "
            f"rep={item['repetition_penalty']:.3f}, "
            f"len={item['length_score']:.3f}"
        )
    print()

DIRECTION: standard_to_genz
INPUT: I am really tired after work.

BEST CANDIDATE:
I'm totally drained after work, fr fr.

RANKED CANDIDATES:
1. I'm totally drained after work, fr fr. | score=0.635, sem=0.571, slang=2, slang_norm=1.000, copy=0.200, copy_pen=0.000, rewrite=0.700, rep=0.143, len=0.833
2. I'm hella drained after work, fr fr. | score=0.617, sem=0.510, slang=3, slang_norm=1.000, copy=0.200, copy_pen=0.000, rewrite=0.700, rep=0.143, len=0.833
3. I'm totally drained after work, fr. | score=0.571, sem=0.618, slang=1, slang_norm=0.500, copy=0.200, copy_pen=0.000, rewrite=0.700, rep=0.000, len=1.000
4. I'm deadass exhausted after grind, fr. | score=0.502, sem=0.379, slang=2, slang_norm=1.000, copy=0.091, copy_pen=0.000, rewrite=0.100, rep=0.000, len=1.000
5. I'm hella drained after grindin' all day, fr fr. | score=0.401, sem=0.312, slang=3, slang_norm=1.000, copy=0.077, copy_pen=0.000, rewrite=0.100, rep=0.111, len=0.500
6. I'm dead tired after grinding all day. | score=0.306, se

In [75]:
# Step 5E: Build predictions for raw vs reranked BART on a validation subset

def build_raw_vs_reranked_predictions(
    df,
    source_col,
    target_col,
    direction="standard_to_genz",
    n_samples=100,
    random_state=42,
    candidate_mode="hybrid",
    num_return_sequences=5
):
    sample_df = df.sample(n=min(n_samples, len(df)), random_state=random_state).copy()

    rows = []

    for _, row in sample_df.iterrows():
        source = row[source_col]
        reference = row[target_col]

        raw_pred = translate_text(source, direction=direction)

        reranked_result = translate_with_reranking(
            source,
            direction=direction,
            candidate_mode=candidate_mode,
            num_return_sequences=num_return_sequences
        )
        reranked_pred = reranked_result["best_candidate"]

        rows.append({
            "source": source,
            "reference": reference,
            "raw_bart": raw_pred,
            "reranked_bart": reranked_pred
        })

    return pd.DataFrame(rows)

In [76]:
# Standard -> Gen Z subset
raw_vs_reranked_std2genz = build_raw_vs_reranked_predictions(
    df=val_df,
    source_col="plain",
    target_col="gen_z",
    direction="standard_to_genz",
    n_samples=100,
    random_state=42,
    candidate_mode="hybrid",
    num_return_sequences=5
)

raw_vs_reranked_std2genz.head()

,source,reference,raw_bart,reranked_bart
0,"I'm really tired today, I just want to relax and do nothing.","I'm so drained today, just wanna chill and do nada.","I'm hella drained today, just tryna chill and do nothing.","I'm hella drained today, just wanna chill and do nothing."
1,I'm just really tired today and all I want to do is relax at home.,"I'm totally drained today and just wanna chill at home, fr.",I'm hella drained today and all I wanna do is chill at home.,I'm hella drained today and all I wanna do is chill at home.
2,"Hey, want to grab some coffee later?","Yo, down to get some coffee later?","Yo, wanna snag some coffee later?","Yo, you down to snag some coffee later?"
3,I'm going to grab some coffee. Do you want to come?,"I'm finna get some coffee, yolo? Do you wanna roll?",I'm finna snag some coffee. You tryna slide?,I'm about to snag some coffee. You tryna slide?
4,I'm so tired after a long day at work.,"I'm so drained after a mega long shift, fr.",I'm totally drained after a long day at the grind.,"I'm hella drained after a long day at work, fr."


In [77]:
# Gen Z -> standard subset
raw_vs_reranked_genz2std = build_raw_vs_reranked_predictions(
    df=val_df,
    source_col="gen_z",
    target_col="plain",
    direction="genz_to_standard",
    n_samples=100,
    random_state=42,
    candidate_mode="hybrid",
    num_return_sequences=5
)

raw_vs_reranked_genz2std.head()

,source,reference,raw_bart,reranked_bart
0,"I'm so drained today, just wanna chill and do nada.","I'm really tired today, I just want to relax and do nothing.","I'm really tired today, I just want to relax and do nothing.","I'm really tired today, I just want to relax and do nada."
1,"I'm totally drained today and just wanna chill at home, fr.",I'm just really tired today and all I want to do is relax at home.,I'm really tired today and just want to relax at home.,I'm really tired today and just want to relax at home.
2,"Yo, down to get some coffee later?","Hey, want to grab some coffee later?","Hey, do you want to grab some coffee later?","Hey, do you want to grab coffee later?"
3,"I'm finna get some coffee, yolo? Do you wanna roll?",I'm going to grab some coffee. Do you want to come?,I'm going to grab some coffee. Do you want to come?,I'm going to grab some coffee. Want to go?
4,"I'm so drained after a mega long shift, fr.",I'm so tired after a long day at work.,I'm really tired after a long day at work.,I'm so tired after a long day at work.


In [79]:
!pip -q install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00


In [80]:
# Metric setup / helpers for reranking experiments
import evaluate
from bert_score import score

rouge = evaluate.load("rouge")

# slang vocab setup
slang_vocab_single = set()
slang_vocab_phrases = []

for s in slang_terms:
    s_clean = s.lower().strip()
    if " " in s_clean:
        slang_vocab_phrases.append(s_clean)
    else:
        slang_vocab_single.add(s_clean)

def slang_score_eval(text):
    text_clean = text.lower().strip()

    tokens = re.findall(r"\b[\w']+\b", text_clean)
    if not tokens:
        return 0.0

    single_count = sum(1 for t in tokens if t in slang_vocab_single)

    phrase_count = 0
    for phrase in slang_vocab_phrases:
        if phrase in text_clean:
            phrase_count += 1

    return (single_count + 2 * phrase_count) / len(tokens)

def length_ratio(preds, inputs):
    return sum(len(p.split()) / len(i.split()) for p, i in zip(preds, inputs)) / len(preds)

In [81]:
def evaluate_prediction_table(df, pred_col, reference_col, source_col=None):
    predictions = df[pred_col].fillna("").tolist()
    references = df[reference_col].fillna("").tolist()

    results = {}

    # ROUGE
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    results["rouge1"] = rouge_scores["rouge1"]
    results["rouge2"] = rouge_scores["rouge2"]
    results["rougeL"] = rouge_scores["rougeL"]

    # BERTScore
    P, R, F1 = score(
        predictions,
        references,
        lang="en",
        verbose=True
    )
    results["bertscore_f1"] = float(F1.mean().item())

    # Average slang score of predictions
    results["avg_slang_score"] = float(np.mean([slang_score_eval(p) for p in predictions]))

    # Average length ratio relative to reference
    ref_lens = [max(1, len(normalize_text(r).split())) for r in references]
    pred_lens = [max(1, len(normalize_text(p).split())) for p in predictions]
    results["avg_length_ratio_vs_ref"] = float(np.mean([
        p / r for p, r in zip(pred_lens, ref_lens)
    ]))

    # Optional: average semantic similarity to source
    if source_col is not None:
        sources = df[source_col].fillna("").tolist()
        results["avg_semantic_similarity_to_source"] = float(np.mean([
            semantic_similarity(src, pred) for src, pred in zip(sources, predictions)
        ]))

    return results

In [82]:
# Standard -> Gen Z metrics
std2genz_raw_metrics = evaluate_prediction_table(
    raw_vs_reranked_std2genz,
    pred_col="raw_bart",
    reference_col="reference",
    source_col="source"
)

std2genz_reranked_metrics = evaluate_prediction_table(
    raw_vs_reranked_std2genz,
    pred_col="reranked_bart",
    reference_col="reference",
    source_col="source"
)

# Gen Z -> standard metrics
genz2std_raw_metrics = evaluate_prediction_table(
    raw_vs_reranked_genz2std,
    pred_col="raw_bart",
    reference_col="reference",
    source_col="source"
)

genz2std_reranked_metrics = evaluate_prediction_table(
    raw_vs_reranked_genz2std,
    pred_col="reranked_bart",
    reference_col="reference",
    source_col="source"
)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/3 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 0.61 seconds, 164.15 sentences/sec


In [83]:
comparison_metrics_df = pd.DataFrame([
    {"direction": "standard_to_genz", "system": "raw_bart", **std2genz_raw_metrics},
    {"direction": "standard_to_genz", "system": "reranked_bart", **std2genz_reranked_metrics},
    {"direction": "genz_to_standard", "system": "raw_bart", **genz2std_raw_metrics},
    {"direction": "genz_to_standard", "system": "reranked_bart", **genz2std_reranked_metrics},
])

comparison_metrics_df

,direction,system,rouge1,rouge2,rougeL,bertscore_f1,avg_slang_score,avg_length_ratio_vs_ref,avg_semantic_similarity_to_source
0,standard_to_genz,raw_bart,0.704777,0.517198,0.698725,0.952315,0.084490,1.002438,0.630166
1,standard_to_genz,reranked_bart,0.659605,0.452318,0.650830,0.944606,0.093617,1.049724,0.649038
2,genz_to_standard,raw_bart,0.867544,0.752380,0.860106,0.983755,0.035053,1.007101,0.618783
3,genz_to_standard,reranked_bart,0.818249,0.654908,0.811529,0.977043,0.036278,0.985076,0.650774


In [84]:
# Qualitative comparison: raw BART vs reranked BART

def build_comparison_table(
    df,
    source_col,
    target_col,
    direction,
    n_samples=25,
    random_state=42
):
    sample_df = df.sample(n=min(n_samples, len(df)), random_state=random_state).copy()

    rows = []

    for _, row in sample_df.iterrows():
        source = row[source_col]
        reference = row[target_col]

        raw_pred = translate_text(source, direction=direction)

        reranked_pred = translate_with_reranking(
            source,
            direction=direction
        )["best_candidate"]

        rows.append({
            "source": source,
            "reference": reference,
            "raw_bart": raw_pred,
            "reranked_bart": reranked_pred
        })

    return pd.DataFrame(rows)

In [85]:
comparison_std_to_genz = build_comparison_table(
    val_df, "plain", "gen_z", "standard_to_genz"
)

comparison_genz_to_std = build_comparison_table(
    val_df, "gen_z", "plain", "genz_to_standard"
)

In [86]:
def print_examples(df, n=10):
    for i, (_, row) in enumerate(df.head(n).iterrows(), 1):
        print("=" * 100)
        print(f"EXAMPLE {i}")
        print("SOURCE:   ", row["source"])
        print("REFERENCE:", row["reference"])
        print("RAW BART: ", row["raw_bart"])
        print("RERANKED: ", row["reranked_bart"])
        print()

print_examples(comparison_std_to_genz, 10)
print_examples(comparison_genz_to_std, 10)

EXAMPLE 1
SOURCE:    I'm really tired today, I just want to relax and do nothing.
REFERENCE: I'm so drained today, just wanna chill and do nada.
RAW BART:  I'm hella drained today, just tryna chill and do nothing.
RERANKED:  I'm hella drained today, just wanna chill and do nothing.

EXAMPLE 2
SOURCE:    I'm just really tired today and all I want to do is relax at home.
REFERENCE: I'm totally drained today and just wanna chill at home, fr.
RAW BART:  I'm hella drained today and all I wanna do is chill at home.
RERANKED:  I'm hella drained today and all I wanna do is chill at home.

EXAMPLE 3
SOURCE:    Hey, want to grab some coffee later?
REFERENCE: Yo, down to get some coffee later?
RAW BART:  Yo, wanna snag some coffee later?
RERANKED:  Yo, you down to snag some coffee later?

EXAMPLE 4
SOURCE:    I'm going to grab some coffee. Do you want to come?
REFERENCE: I'm finna get some coffee, yolo? Do you wanna roll?
RAW BART:  I'm finna snag some coffee. You tryna slide?
RERANKED:  I'm abou

## Step 6 Eval Metrics (on val and baseline for now)


In [ ]:
# Compare automatic metrics (BART)
import evaluate

rouge = evaluate.load("rouge")

# Split validation set by direction
val_std_to_genz = val_df[val_df["direction"] == "standard_to_genz"].reset_index(drop=True)
val_genz_to_std = val_df[val_df["direction"] == "genz_to_standard"].reset_index(drop=True)

print("standard_to_genz validation size:", len(val_std_to_genz))
print("genz_to_standard validation size:", len(val_genz_to_std))



standard_to_genz validation size: 114
genz_to_standard validation size: 115


In [ ]:
bart_predictions_std_to_genz = []
references_std_to_genz = []

for _, row in val_std_to_genz.iterrows():
    pred = translate_text(row["plain"], direction="standard_to_genz")
    bart_predictions_std_to_genz.append(pred.strip())
    references_std_to_genz.append(row["gen_z"].strip())

bart_results_std_to_genz = rouge.compute(
    predictions=bart_predictions_std_to_genz,
    references=references_std_to_genz,
    use_stemmer=True
)

bart_results_std_to_genz

{'rouge1': np.float64(0.6878666329221768),
 'rouge2': np.float64(0.4850828612981578),
 'rougeL': np.float64(0.6748328551756617),
 'rougeLsum': np.float64(0.6753025214699313)}

In [ ]:
bart_predictions_genz_to_std = []
references_genz_to_std = []

for _, row in val_genz_to_std.iterrows():
    pred = translate_text(row["gen_z"], direction="genz_to_standard")
    bart_predictions_genz_to_std.append(pred.strip())
    references_genz_to_std.append(row["plain"].strip())

bart_results_genz_to_std = rouge.compute(
    predictions=bart_predictions_genz_to_std,
    references=references_genz_to_std,
    use_stemmer=True
)

bart_results_genz_to_std

{'rouge1': np.float64(0.8368302452784082),
 'rouge2': np.float64(0.7074383112348577),
 'rougeL': np.float64(0.827556854670108),
 'rougeLsum': np.float64(0.8298236811661812)}

In [ ]:
rule_predictions_std_to_genz = []

for _, row in val_std_to_genz.iterrows():
    pred = rule_based_genz(row["plain"])
    rule_predictions_std_to_genz.append(pred.strip())

rule_results_std_to_genz = rouge.compute(
    predictions=rule_predictions_std_to_genz,
    references=references_std_to_genz,
    use_stemmer=True
)

rule_results_std_to_genz

{'rouge1': np.float64(0.6008608022143286),
 'rouge2': np.float64(0.34152635914515717),
 'rougeL': np.float64(0.5838925088769533),
 'rougeLsum': np.float64(0.583885490942561)}

In [ ]:
metrics_std_to_genz_df = pd.DataFrame([
    {
        "direction": "standard_to_genz",
        "model": "rule_based",
        "rouge1": rule_results_std_to_genz["rouge1"],
        "rouge2": rule_results_std_to_genz["rouge2"],
        "rougeL": rule_results_std_to_genz["rougeL"]
    },
    {
        "direction": "standard_to_genz",
        "model": "bart",
        "rouge1": bart_results_std_to_genz["rouge1"],
        "rouge2": bart_results_std_to_genz["rouge2"],
        "rougeL": bart_results_std_to_genz["rougeL"]
    }
])

metrics_std_to_genz_df

,direction,model,rouge1,rouge2,rougeL
0,standard_to_genz,rule_based,0.600861,0.341526,0.583893
1,standard_to_genz,bart,0.687867,0.485083,0.674833


In [ ]:
metrics_genz_to_std_df = pd.DataFrame([
    {
        "direction": "genz_to_standard",
        "model": "bart",
        "rouge1": bart_results_genz_to_std["rouge1"],
        "rouge2": bart_results_genz_to_std["rouge2"],
        "rougeL": bart_results_genz_to_std["rougeL"]
    }
])

metrics_genz_to_std_df

,direction,model,rouge1,rouge2,rougeL
0,genz_to_standard,bart,0.83683,0.707438,0.827557


In [ ]:
!pip -q install bert-score

from bert_score import score

In [ ]:
P_std, R_std, F1_std = score(
    bart_predictions_std_to_genz,
    references_std_to_genz,
    lang="en",
    verbose=True
)

bertscore_std_to_genz = {
    "direction": "standard_to_genz",
    "precision": P_std.mean().item(),
    "recall": R_std.mean().item(),
    "f1": F1_std.mean().item()
}

bertscore_std_to_genz

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 0.92 seconds, 123.72 sentences/sec


{'direction': 'standard_to_genz',
 'precision': 0.9512932896614075,
 'recall': 0.9461306929588318,
 'f1': 0.9486297965049744}

In [ ]:
P_rev, R_rev, F1_rev = score(
    bart_predictions_genz_to_std,
    references_genz_to_std,
    lang="en",
    verbose=True
)

bertscore_genz_to_std = {
    "direction": "genz_to_standard",
    "precision": P_rev.mean().item(),
    "recall": R_rev.mean().item(),
    "f1": F1_rev.mean().item()
}

bertscore_genz_to_std

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/4 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/2 [00:00<?, ?it/s]

done in 0.92 seconds, 124.51 sentences/sec


{'direction': 'genz_to_standard',
 'precision': 0.979246199131012,
 'recall': 0.9802459478378296,
 'f1': 0.9797206521034241}

In [ ]:
# single-word slang tokens
slang_vocab_single = set()
slang_vocab_phrases = []

for s in slang_terms:
    s_clean = s.lower().strip()
    if " " in s_clean:
        slang_vocab_phrases.append(s_clean)
    else:
        slang_vocab_single.add(s_clean)

def slang_score(text):
    text_clean = text.lower().strip()

    tokens = re.findall(r"\b[\w']+\b", text_clean)
    if not tokens:
        return 0.0

    single_count = sum(1 for t in tokens if t in slang_vocab_single)

    phrase_count = 0
    for phrase in slang_vocab_phrases:
        if phrase in text_clean:
            phrase_count += 1

    return (single_count + 2 * phrase_count) / len(tokens)

bart_slang_scores_std_to_genz = [slang_score(p) for p in bart_predictions_std_to_genz]
rule_slang_scores_std_to_genz = [slang_score(p) for p in rule_predictions_std_to_genz]

print("BART slang score (standard_to_genz):", sum(bart_slang_scores_std_to_genz) / len(bart_slang_scores_std_to_genz))
print("RULE slang score (standard_to_genz):", sum(rule_slang_scores_std_to_genz) / len(rule_slang_scores_std_to_genz))

BART slang score (standard_to_genz): 0.07514459224985541
RULE slang score (standard_to_genz): 0.07497819661132354


In [ ]:
def length_ratio(preds, inputs):
    return sum(len(p.split()) / len(i.split()) for p, i in zip(preds, inputs)) / len(preds)

# standard -> Gen Z
bart_len_std_to_genz = length_ratio(bart_predictions_std_to_genz, val_std_to_genz["plain"])
rule_len_std_to_genz = length_ratio(rule_predictions_std_to_genz, val_std_to_genz["plain"])

print("BART length ratio (standard_to_genz):", bart_len_std_to_genz)
print("RULE length ratio (standard_to_genz):", rule_len_std_to_genz)

# Gen Z -> standard
bart_len_genz_to_std = length_ratio(bart_predictions_genz_to_std, val_genz_to_std["gen_z"])

print("BART length ratio (genz_to_standard):", bart_len_genz_to_std)

BART length ratio (standard_to_genz): 0.9170684700289963
RULE length ratio (standard_to_genz): 0.9945772794457004
BART length ratio (genz_to_standard): 1.0765153686892819


## Step 7
